# Studi Kasus: Atribusi Penyebab Langsung Peringatan Deforestasi di Provinsi Bengkulu Menggunakan Citra Satelit (Sentinel-1 & Sentinel-2) dan Kecerdasan Buatan

**Tugas Kuliah Praktisi Informatika**
Materi: *Attributing Direct Drivers to Deforestation Alerts with Satellite Imagery and AI*
Narasumber: Dr. Bart Slagter — Wageningen University & Research (WUR), Belanda
Program Studi S1 Teknik Informatika, Fakultas Teknik, Universitas Bengkulu

| Identitas | |
|---|---|
| Nama | ........................................ |
| NPM | ........................................ |
| Kelas | ........................................ |

> **Cara menjalankan:** buka notebook ini di Google Colab, lalu pilih menu **Runtime → Run all**. Semua pustaka yang dipakai (NumPy, pandas, SciPy, scikit-learn, TensorFlow/Keras, Matplotlib, Seaborn, Folium) sudah tersedia di Colab sehingga tidak perlu instalasi tambahan. Waktu eksekusi ± 5–10 menit pada runtime CPU standar; pelatihan CNN lebih cepat bila memakai GPU (**Runtime → Change runtime type → T4 GPU**).

### Daftar Isi
1. Latar Belakang
2. Deskripsi Studi Kasus
3. Data dan Asumsi
4. Karakteristik Penyebab Langsung Deforestasi
5. Metodologi
6. Implementasi Kode
7. Analisis Hasil
8. Keterbatasan dan Pengembangan Lanjutan
9. Kesimpulan
10. Referensi
11. Lampiran: Pengambilan Data Riil dengan Google Earth Engine

## 1. Latar Belakang

Sistem peringatan deforestasi berbasis satelit seperti **RADD** (*RAdar for Detecting Deforestation*, berbasis Sentinel-1) dan **GLAD** (berbasis Landsat/Sentinel-2) yang ditampilkan di **Global Forest Watch** sudah mampu menjawab *di mana* dan *kapan* tutupan hutan hilang. Namun, petugas kehutanan juga perlu mengetahui ***mengapa*** hutan hilang. Penanganan bukaan tambang emas tanpa izin tentu berbeda dengan penanganan longsor alami atau kebun rakyat di tepi hutan.

Dalam kuliah praktisi, Dr. Bart Slagter memaparkan bahwa penyebab langsung (*direct drivers*) — perkebunan skala besar, pertanian rakyat, penebangan kayu, pembangunan infrastruktur, pertambangan, kebakaran, bencana alam, dan dinamika air — meninggalkan "jejak" yang berbeda pada citra satelit: luas dan bentuk bukaan, respons spektral vegetasi, respons hamburan balik radar, serta pola perubahan dari waktu ke waktu. Pola ini dapat dipelajari oleh model *Computer Vision* dan *Deep Learning*.

Dua sumber data utama yang saling melengkapi:
- **Sentinel-2 (optik, multispektral)** — kaya informasi spektral vegetasi, tanah, air, dan bekas terbakar, tetapi terhalang awan.
- **Sentinel-1 (radar SAR, polarisasi VV & VH)** — menembus awan dan bekerja siang–malam, sangat penting di wilayah tropis seperti Bengkulu yang curah hujannya tinggi.

Penelitian Slagter dkk. (2023) membuktikan pendekatan ini secara nyata: *convolutional neural network* yang dilatih dengan data Sentinel-1 dan Sentinel-2 dipakai untuk mengklasifikasikan penyebab alert RADD di Suriname, Republik Kongo, dan RD Kongo. Komposit Sentinel-2 beberapa bulan setelah gangguan memberi hasil paling akurat, sementara Sentinel-1 membantu klasifikasi yang lebih cepat ketika awan menutupi wilayah.

Provinsi Bengkulu memiliki hutan pegunungan Bukit Barisan, termasuk sebagian kawasan **Taman Nasional Kerinci Seblat (TNKS)** dan **Taman Nasional Bukit Barisan Selatan (TNBBS)**, yang menghadapi tekanan dari perluasan kebun, perambahan, penambangan, dan bencana hidrometeorologi. Studi kasus ini menerapkan konsep dari kuliah tersebut ke konteks Bengkulu.

## 2. Deskripsi Studi Kasus

**Skenario.** Sebuah unit pemantauan hutan tingkat provinsi di Bengkulu menerima ±150 alert kehilangan tutupan hutan dalam satu periode pemantauan. Jumlah tim patroli terbatas (misalnya 3 tim, masing-masing hanya mampu memverifikasi ±10 lokasi per minggu), sehingga tidak semua alert dapat langsung dicek ke lapangan.

**Permasalahan yang diselesaikan:**
1. Alert hanya memberi lokasi dan tanggal, tanpa informasi penyebab gangguan.
2. Tutupan awan yang tinggi di Bengkulu membuat citra optik sering tidak tersedia, sehingga klasifikasi yang hanya bergantung pada citra optik menjadi tidak andal.
3. Tidak ada mekanisme objektif untuk menentukan alert mana yang perlu diperiksa lebih dahulu.

**Tujuan:**
1. Membangun model AI yang mengklasifikasikan setiap alert ke dalam **8 kelas penyebab langsung** berdasarkan citra Sentinel-1, Sentinel-2, dan deret waktu.
2. Membandingkan kinerja data optik saja, radar saja, dan fusi keduanya, termasuk ketahanannya terhadap tutupan awan.
3. Membandingkan model *machine learning* klasik (Random Forest) dengan *deep learning* (CNN multimodal) serta gabungannya (*ensemble*).
4. Menyusun **skor prioritas** pemeriksaan lapangan dan menandai alert berkeyakinan rendah yang wajib diverifikasi.
5. Menyajikan hasil dalam **peta interaktif** dan **dasbor** pendukung keputusan, serta mengekspor data ke format CSV dan GeoJSON.

**Batasan:**
- Data citra dan alert dibangkitkan secara sintetis (disimulasikan) mengikuti karakteristik fisik Sentinel-1/2; alasannya dijelaskan pada Bagian 3.
- Lokasi alert dan status kawasan pada peta adalah **simulasi ilustratif**, bukan batas kawasan resmi.
- Satu alert diasumsikan memiliki satu penyebab dominan.

## 3. Data dan Asumsi

Data berlabel penyebab deforestasi untuk Bengkulu tidak tersedia secara terbuka, dan pembuatannya memerlukan interpretasi visual serta survei lapangan. Agar seluruh alur kerja dapat dijalankan dan dievaluasi secara utuh di Google Colab tanpa akun atau unduhan tambahan, notebook ini **membangkitkan data sintetis yang meniru karakteristik fisik citra satelit**. Lampiran (Bagian 11) memberikan kode untuk mengganti data sintetis dengan data riil dari Google Earth Engine.

| Komponen data | Isi | Keterangan |
|---|---|---|
| Mask alert | Piksel yang terdeteksi berubah | Diasumsikan berasal dari sistem alert (RADD/GLAD) |
| Patch Sentinel-2 sebelum & sesudah | Band B2, B3, B4, B8, B11, B12 (reflektansi) | Patch 32×32 piksel, resolusi 20 m (640 m × 640 m); citra sesudah dapat tertutup awan |
| Patch Sentinel-1 sebelum & sesudah | Polarisasi VV dan VH (dB) | Mengandung *speckle* radar, tidak terpengaruh awan |
| Deret waktu bulanan | NDVI, NBR (optik) dan VH (radar) | 24 bulan (Okt 2024 – Sep 2026); bulan berawan bernilai kosong (NaN) |
| Metadata | Lokasi, kabupaten, status kawasan, tanggal alert | Untuk prioritisasi dan pemetaan |

**Asumsi simulasi:**
- Nilai reflektansi dan hamburan balik radar setiap jenis permukaan (hutan, tanah terbuka, bekas terbakar, air, tanaman muda, jalan, area tambang, batuan longsoran) mengikuti kisaran umum di literatur penginderaan jauh, ditambah variasi acak antarsampel.
- Bentuk bukaan, jenis permukaan sesudah gangguan, musim kejadian, dan pola pemulihan vegetasi dibuat berbeda untuk setiap penyebab, tetapi **sengaja dibuat saling tumpang tindih** (misalnya pertanian rakyat kadang dibuka dengan membakar, perkebunan kadang dibakar saat pembukaan lahan, dan jalan logging mirip jalan infrastruktur) agar masalah klasifikasinya realistis dan tidak terlalu mudah.
- Peluang awan pada citra sesudah = 60%, dengan tutupan 10–80% patch; peluang bulan berawan pada deret waktu ±35% (lebih tinggi pada musim hujan).

## 4. Karakteristik Penyebab Langsung Deforestasi

| No | Penyebab | Bentuk & ukuran bukaan | Respons optik (Sentinel-2) | Respons radar (Sentinel-1) | Pola waktu |
|---|---|---|---|---|---|
| 1 | Perkebunan skala besar | Luas, tepi lurus, persegi/blok teratur | NDVI turun tajam, tanah terbuka atau tanaman muda | VH turun karena kehilangan volume tajuk | Turun tajam, vegetasi naik perlahan setelah ±6 bulan (tanaman ditanam) |
| 2 | Pertanian rakyat | Kecil–sedang, tidak teratur, terpencar | Campuran tanah, tanaman, kadang bekas bakar | Penurunan VH sedang | Pemulihan cepat dan berfluktuasi mengikuti siklus tanam |
| 3 | Penebangan kayu | Celah-celah kecil tersebar + jalan sarad tipis | Penurunan NDVI kecil dan tidak merata | Perubahan kecil | Pulih cepat (3–6 bulan) karena tajuk menutup kembali |
| 4 | Infrastruktur (jalan) | Linier, memanjang, lebar sempit | Permukaan terang, tanah padat | VV/VH rendah (permukaan halus) | Tidak pulih |
| 5 | Pertambangan | Kompak, tidak teratur, ada kolam air | Sangat terang di SWIR, terdapat piksel air | VH sangat rendah, bercampur air | Tidak pulih, cenderung meluas |
| 6 | Kebakaran hutan | Luas, tidak beraturan, organik | NBR turun sangat tajam (NIR turun, SWIR naik) | Penurunan VH sedang | Terjadi pada musim kemarau, pemulihan sebagian |
| 7 | Bencana alam (longsor) | Memanjang mengikuti lereng, sempit | Tanah/batuan terbuka | Permukaan kasar, penurunan VH sedang | Umumnya pada musim hujan, pemulihan lambat |
| 8 | Dinamika air | Pita berkelok (sungai/banjir) | NDWI tinggi, NIR sangat rendah | VV & VH sangat rendah (pantulan spekular) | Berfluktuasi musiman |

## 5. Metodologi

```
 Alert kehilangan hutan (lokasi + tanggal + mask)
          │
          ▼
 Ekstraksi data: patch Sentinel-2 & Sentinel-1 (sebelum/sesudah) + deret waktu bulanan
          │
          ▼
 Rekayasa fitur ─ bentuk (luas, kekompakan, elongasi, jumlah komponen)
                ─ spektral optik (NDVI, NBR, NDWI, NDMI, kecerahan)
                ─ radar (VV, VH, rasio VH–VV, fraksi air radar)
                ─ temporal (besar penurunan, laju pemulihan, musim kejadian)
          │
          ├──► Model 1: Random Forest (fitur hasil rekayasa)
          └──► Model 2: CNN multimodal (cabang Conv2D untuk citra + Conv1D untuk deret waktu)
          │
          ▼
 Ensemble probabilitas ─► kelas penyebab + tingkat keyakinan
          │
          ▼
 Skor prioritas (risiko penyebab, status kawasan, luas, keyakinan)
          │
          ▼
 Peta interaktif + dasbor + ekspor CSV/GeoJSON ─► verifikasi lapangan
```

**Pembagian data:** 70% latih, 15% validasi (untuk *early stopping* CNN), 15% uji (stratified).
**Metrik evaluasi:** akurasi keseluruhan (*overall accuracy*), **Macro-F1** (rata-rata F1 semua kelas sehingga kelas kecil tetap diperhitungkan), *precision*/*recall* per kelas, dan *confusion matrix*.

## 6. Implementasi Kode

### 6.1 Import pustaka dan konfigurasi

In [ ]:
# ============================================================
# 6.1 Import pustaka dan konfigurasi umum
# ============================================================
import warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import ndimage

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
from sklearn.inspection import permutation_importance

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

PATCH = 32                                # ukuran patch (piksel)
RESOLUSI = 20                             # meter per piksel -> 640 m x 640 m
LUAS_PIKSEL_HA = RESOLUSI**2 / 10_000     # 0,04 ha per piksel
N_BULAN = 24
TANGGAL = pd.date_range("2024-10-01", periods=N_BULAN, freq="MS")   # Okt 2024 - Sep 2026

KELAS = ["Perkebunan skala besar", "Pertanian rakyat", "Penebangan kayu",
         "Infrastruktur (jalan)", "Pertambangan", "Kebakaran hutan",
         "Bencana alam (longsor)", "Dinamika air"]
KELAS_SINGKAT = ["Perkebunan", "Pertanian", "Penebangan", "Infrastruktur",
                 "Tambang", "Kebakaran", "Longsor", "Air"]
N_KELAS = len(KELAS)
WARNA_KELAS = ["#e6ab02", "#66a61e", "#a6761d", "#666666",
               "#7570b3", "#d7301f", "#e7298a", "#1f78b4"]
BAND_S2 = ["B2", "B3", "B4", "B8", "B11", "B12"]
BAND_S1 = ["VV", "VH"]

print(f"Patch {PATCH}x{PATCH} piksel @ {RESOLUSI} m  -> {PATCH*RESOLUSI} m x {PATCH*RESOLUSI} m")
print(f"Periode deret waktu: {TANGGAL[0]:%b %Y} - {TANGGAL[-1]:%b %Y} ({N_BULAN} bulan)")
print("Kelas penyebab:", ", ".join(KELAS))

### 6.2 Pustaka spektral jenis permukaan

Nilai rujukan reflektansi Sentinel-2 (0–1) dan hamburan balik Sentinel-1 (dB) untuk setiap jenis permukaan. Nilai inilah yang membedakan "tanda tangan" setiap penyebab pada citra.

In [ ]:
# ============================================================
# 6.2 Pustaka spektral (B2, B3, B4, B8, B11, B12 | VV, VH dB)
# ============================================================
SPEKTRAL = {
    "hutan":    [0.020, 0.040, 0.025, 0.300, 0.150, 0.070,  -7.0, -12.5],
    "tanah":    [0.080, 0.110, 0.150, 0.220, 0.300, 0.250, -10.0, -18.0],
    "terbakar": [0.030, 0.040, 0.050, 0.100, 0.180, 0.160,  -9.5, -16.5],
    "air":      [0.050, 0.060, 0.040, 0.020, 0.010, 0.005, -20.0, -26.0],
    "tanaman":  [0.035, 0.070, 0.055, 0.280, 0.210, 0.120,  -8.5, -14.5],
    "jalan":    [0.100, 0.120, 0.140, 0.200, 0.280, 0.240, -11.0, -19.5],
    "tambang":  [0.120, 0.150, 0.200, 0.250, 0.350, 0.300, -12.0, -20.0],
    "batuan":   [0.110, 0.130, 0.150, 0.230, 0.320, 0.270,  -9.0, -16.0],
}
SPEKTRAL = {k: np.array(v, dtype=np.float32) for k, v in SPEKTRAL.items()}

tabel_spektral = pd.DataFrame(SPEKTRAL, index=BAND_S2 + [b + " (dB)" for b in BAND_S1]).T
s = tabel_spektral
tabel_spektral["NDVI"] = (s["B8"] - s["B4"]) / (s["B8"] + s["B4"])
tabel_spektral["NBR"]  = (s["B8"] - s["B12"]) / (s["B8"] + s["B12"])
tabel_spektral["NDWI"] = (s["B3"] - s["B8"]) / (s["B3"] + s["B8"])
tabel_spektral.round(3)

In [ ]:
# Visualisasi kurva spektral dan posisi radar setiap permukaan
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
panjang_gelombang = [490, 560, 665, 842, 1610, 2190]   # nm (pusat band Sentinel-2)
for nama, v in SPEKTRAL.items():
    ax[0].plot(panjang_gelombang, v[:6], marker="o", label=nama)
    ax[1].scatter(v[6], v[7], s=60)
    ax[1].annotate(nama, (v[6], v[7]), xytext=(4, 3), textcoords="offset points")
ax[0].set(xlabel="Panjang gelombang (nm)", ylabel="Reflektansi",
          title="Kurva spektral Sentinel-2 per jenis permukaan")
ax[0].legend(fontsize=7, ncol=2)
ax[1].set(xlabel="VV (dB)", ylabel="VH (dB)", title="Ruang hamburan balik Sentinel-1")
plt.tight_layout(); plt.show()

### 6.3 Generator bentuk bukaan lahan

Setiap penyebab memiliki geometri khas: blok persegi (perkebunan), petak kecil tidak beraturan (pertanian rakyat), celah tersebar + jalan sarad (penebangan), garis (jalan), gumpalan kompak dengan kolam (tambang), area organik luas (kebakaran), elips memanjang searah lereng (longsor), dan pita berkelok (sungai).

In [ ]:
# ============================================================
# 6.3 Fungsi pembentuk geometri bukaan
# ============================================================
YY, XX = np.mgrid[:PATCH, :PATCH]

def noise_halus(rng, sigma=2.0):
    """Derau spasial halus (ternormalisasi) untuk membuat tepi tidak beraturan / tekstur."""
    n = ndimage.gaussian_filter(rng.normal(size=(PATCH, PATCH)), sigma)
    return (n - n.mean()) / (n.std() + 1e-9)

def bentuk_blob(rng, cy, cx, r, ketidakteraturan=0.6):
    d = np.hypot(YY - cy, XX - cx) / r
    return d + 0.35 * ketidakteraturan * noise_halus(rng, 2.5) < 1.0

def _rotasi(cy, cx, sudut):
    a = np.deg2rad(sudut)
    y, x = YY - cy, XX - cx
    u = x * np.cos(a) + y * np.sin(a)      # sumbu sepanjang objek
    v = -x * np.sin(a) + y * np.cos(a)     # sumbu melintang
    return u, v

def bentuk_persegi(cy, cx, tinggi, lebar, sudut):
    u, v = _rotasi(cy, cx, sudut)
    return (np.abs(u) <= lebar / 2) & (np.abs(v) <= tinggi / 2)

def bentuk_elips(cy, cx, panjang, lebar, sudut):
    u, v = _rotasi(cy, cx, sudut)
    return (u / (panjang / 2))**2 + (v / (lebar / 2))**2 <= 1

def bentuk_garis(rng, lebar, lengkung=0.0, amplitudo=0.0, periode=16):
    """Garis/pita melintasi patch; bisa melengkung (jalan) atau berkelok (sungai)."""
    u, v = _rotasi(*rng.uniform(10, 22, 2), rng.uniform(0, 180))
    v = v - lengkung * u**2 / PATCH - amplitudo * np.sin(2 * np.pi * u / periode + rng.uniform(0, 2 * np.pi))
    return np.abs(v) <= lebar / 2

def bentuk_celah_tersebar(rng, n):
    m = np.zeros((PATCH, PATCH), bool)
    for _ in range(n):
        m |= np.hypot(YY - rng.uniform(2, 30), XX - rng.uniform(2, 30)) <= rng.uniform(0.6, 1.6)
    return m

### 6.4 Generator sampel alert (citra Sentinel-2, Sentinel-1, dan deret waktu)

Fungsi `buat_sampel()` menghasilkan satu alert lengkap:
1. **Pola penyebab** → mask alert + lapisan permukaan sesudah gangguan.
2. **Citra sebelum** → hutan dengan tekstur tajuk (kadang ada bukaan lama di sekitarnya).
3. **Citra sesudah** → permukaan baru dicampur dengan hutan sesuai fraksi gangguan.
4. **Efek sensor** → variasi radiometrik, derau optik, *speckle* radar, dan **awan** pada citra optik sesudah.
5. **Deret waktu bulanan** NDVI, NBR, VH dengan musim kejadian dan kurva pemulihan khas penyebab; bulan berawan bernilai NaN.

In [ ]:
# ============================================================
# 6.4a Pola spasial dan permukaan setiap penyebab
# ============================================================
def pola_penyebab(kelas, rng):
    """Mengembalikan mask alert (bool) dan daftar lapisan (mask, permukaan, fraksi)."""
    kosong = lambda: np.zeros((PATCH, PATCH), bool)
    lapisan = []
    if kelas == 0:                                    # Perkebunan skala besar
        m, sudut = kosong(), rng.uniform(-30, 30)
        if rng.random() < 0.8:                        # blok teratur
            for _ in range(rng.integers(1, 3)):
                m |= bentuk_persegi(rng.uniform(8, 24), rng.uniform(8, 24),
                                    rng.uniform(8, 24), rng.uniform(10, 28), sudut + rng.normal(0, 4))
        else:                                         # pembukaan bertahap, tepi tidak rapi
            m |= bentuk_blob(rng, rng.uniform(10, 22), rng.uniform(10, 22), rng.uniform(6, 12), 0.9)
        permukaan = rng.choice(["tanah", "tanaman", "terbakar"], p=[0.45, 0.35, 0.2])
        lapisan.append((m, permukaan, rng.uniform(0.75, 1.0)))
        if rng.random() < 0.3:                        # jalan akses kebun
            jalan = bentuk_garis(rng, rng.uniform(1.0, 2.0), lengkung=rng.uniform(0, 0.3))
            m = m | jalan
            lapisan.append((jalan, "jalan", 0.85))
    elif kelas == 1:                                  # Pertanian rakyat
        m = kosong()
        if rng.random() < 0.8:                        # petak tidak beraturan
            for _ in range(rng.integers(1, 4)):
                m |= bentuk_blob(rng, rng.uniform(4, 28), rng.uniform(4, 28), rng.uniform(2.0, 6.0), 1.0)
        else:                                         # petak kecil persegi (ladang tertata)
            for _ in range(rng.integers(1, 3)):
                m |= bentuk_persegi(rng.uniform(5, 27), rng.uniform(5, 27), rng.uniform(4, 9),
                                    rng.uniform(5, 11), rng.uniform(0, 90))
        permukaan = rng.choice(["tanaman", "tanah", "terbakar"], p=[0.45, 0.3, 0.25])
        lapisan.append((m, permukaan, rng.uniform(0.6, 1.0)))
        if rng.random() < 0.2:                        # jalan setapak / jalan desa baru
            jalan = bentuk_garis(rng, rng.uniform(0.8, 1.6), lengkung=rng.uniform(0, 0.4))
            m = m | jalan
            lapisan.append((jalan, "tanah", 0.8))
    elif kelas == 2:                                  # Penebangan kayu (selektif)
        m = kosong()
        if rng.random() < 0.8:                        # celah tajuk tersebar
            m = bentuk_celah_tersebar(rng, rng.integers(6, 22))
            fraksi = np.clip(0.5 + 0.15 * noise_halus(rng, 1.0), 0.25, 0.8)
            lapisan.append((m, "tanah", fraksi))
        if m.sum() == 0 or rng.random() < 0.6:        # jalan sarad / jalan logging
            jalan = bentuk_garis(rng, rng.uniform(0.8, 2.0), lengkung=rng.uniform(0, 0.4))
            m = m | jalan
            lapisan.append((jalan, "jalan", rng.uniform(0.6, 0.95)))
    elif kelas == 3:                                  # Infrastruktur (jalan)
        m = bentuk_garis(rng, rng.uniform(1.2, 3.2), lengkung=rng.uniform(0, 0.5))
        lapisan.append((m, rng.choice(["jalan", "tanah"], p=[0.7, 0.3]), rng.uniform(0.75, 1.0)))
        if rng.random() < 0.35:                       # area kerja / basecamp
            kamp = bentuk_persegi(rng.uniform(6, 26), rng.uniform(6, 26),
                                  rng.uniform(3, 7), rng.uniform(4, 9), rng.uniform(0, 90))
            m = m | kamp
            lapisan.append((kamp, "tanah", 0.9))
    elif kelas == 4:                                  # Pertambangan
        m = bentuk_blob(rng, rng.uniform(10, 22), rng.uniform(10, 22), rng.uniform(3.5, 9), 1.3)
        lapisan.append((m, rng.choice(["tambang", "tanah"], p=[0.75, 0.25]), rng.uniform(0.8, 1.0)))
        for _ in range(rng.integers(0, 4)):           # kolam bekas galian
            kolam = bentuk_blob(rng, rng.uniform(10, 22), rng.uniform(10, 22), rng.uniform(1.2, 2.8), 0.8) & m
            lapisan.append((kolam, "air", 0.9))
        if rng.random() < 0.4:
            jalan = bentuk_garis(rng, 1.0, lengkung=rng.uniform(0, 0.3))
            m = m | jalan
            lapisan.append((jalan, "jalan", 0.8))
    elif kelas == 5:                                  # Kebakaran hutan
        m = bentuk_blob(rng, rng.uniform(8, 24), rng.uniform(8, 24), rng.uniform(3.5, 14), 1.6)
        keparahan = np.clip(rng.uniform(0.55, 0.85) + 0.2 * noise_halus(rng, 1.5), 0.3, 1.0)
        lapisan.append((m, "terbakar", keparahan))
    elif kelas == 6:                                  # Bencana alam (longsor)
        m, sudut = kosong(), rng.uniform(0, 180)
        for _ in range(rng.integers(1, 4)):
            m |= bentuk_elips(rng.uniform(6, 26), rng.uniform(6, 26),
                              rng.uniform(6, 16), rng.uniform(1.8, 4.5), sudut + rng.normal(0, 12))
        lapisan.append((m, rng.choice(["batuan", "tanah"], p=[0.6, 0.4]), rng.uniform(0.7, 1.0)))
    else:                                             # Dinamika air (sungai berpindah / banjir)
        if rng.random() < 0.75:                       # alur sungai berkelok
            m = bentuk_garis(rng, rng.uniform(2, 5), amplitudo=rng.uniform(1, 6), periode=rng.uniform(12, 24))
        else:                                         # genangan banjir / danau musiman
            m = bentuk_blob(rng, rng.uniform(8, 24), rng.uniform(8, 24), rng.uniform(3, 8), 1.2)
        lapisan.append((m, "air", rng.uniform(0.6, 1.0)))
        if rng.random() < 0.5:                        # endapan sedimen di tepi
            tepi = ndimage.binary_dilation(m, iterations=1) & ~m
            m = m | tepi
            lapisan.append((tepi, "tanah", 0.7))
    return m, lapisan


def pilih_bulan_kejadian(kelas, rng):
    """Indeks bulan alert (6..17) dengan pola musim: kebakaran saat kemarau, longsor & air saat hujan."""
    if kelas == 5 and rng.random() < 0.8:
        return int(rng.choice([8, 9, 10, 11]))         # Jun-Sep 2025
    if kelas in (6, 7) and rng.random() < 0.75:
        return int(rng.choice([12, 13, 14, 15]))       # Okt 2025 - Jan 2026
    return int(rng.integers(6, 18))


# Lintasan pasca-gangguan tidak selalu "khas": kebun bisa ditelantarkan, ladang bisa
# dibiarkan menjadi belukar, bekas tebangan bisa dikonversi. Peluang 30% sebuah alert
# mengikuti kurva pemulihan penyebab lain yang berkaitan.
KURVA_ALTERNATIF = {0: [3, 1], 1: [2, 0], 2: [1, 3], 3: [4, 0], 4: [3, 6], 5: [1, 2], 6: [3, 5], 7: [6, 4]}

def kurva_pemulihan(kelas, dt, rng):
    """Fraksi pemulihan menuju kondisi sebelum (0 = tetap terganggu, 1 = pulih penuh)."""
    if rng.random() < 0.3:
        kelas = int(rng.choice(KURVA_ALTERNATIF[kelas]))
    d = np.maximum(dt, 0).astype(float)
    u = rng.uniform
    if kelas == 0:   r = 0.6 * np.clip((d - u(4, 9)) / u(18, 30), 0, 1)
    elif kelas == 1: r = u(0.4, 0.7) * (1 - np.exp(-d / u(2, 4))) + u(0.08, 0.18) * np.sin(2 * np.pi * d / u(4, 7))
    elif kelas == 2: r = 1 - np.exp(-d / u(2, 4))
    elif kelas == 3: r = np.full_like(d, u(0, 0.08))
    elif kelas == 4: r = -u(0.1, 0.3) * np.minimum(d / 6, 1)
    elif kelas == 5: r = u(0.4, 0.75) * (1 - np.exp(-d / u(3, 6)))
    elif kelas == 6: r = u(0.15, 0.35) * (1 - np.exp(-d / u(6, 12)))
    else:            r = u(0.2, 0.6) * (0.5 - 0.5 * np.cos(2 * np.pi * d / 12))
    r[dt < 0] = 0
    return r

In [ ]:
# ============================================================
# 6.4b Fungsi indeks spektral dan pembangkit satu sampel alert
# ============================================================
def indeks_spektral(s2):
    B2, B3, B4, B8, B11, B12 = np.moveaxis(s2, -1, 0)
    e = 1e-6
    return {"NDVI":   (B8 - B4) / (B8 + B4 + e),
            "NBR":    (B8 - B12) / (B8 + B12 + e),
            "NDWI":   (B3 - B8) / (B3 + B8 + e),
            "NDMI":   (B8 - B11) / (B8 + B11 + e),
            "BRIGHT": (B2 + B3 + B4) / 3}

def _speckle(db, rng, looks=5):
    lin = 10 ** (db / 10) * rng.gamma(looks, 1 / looks, size=db.shape)
    return 10 * np.log10(lin)

def buat_sampel(kelas, rng, p_awan=0.6, tutupan_awan=None, p_ts_hilang=0.35):
    # 1) pola penyebab
    while True:
        mask, lapisan = pola_penyebab(kelas, rng)
        if mask.sum() >= 4:
            break
    # 2) kondisi sebelum: hutan bertekstur (+ kadang bukaan lama)
    def variasi(nama, sd_optik=0.12, sd_radar=1.0):
        """Spektrum permukaan dengan variasi antarlokasi (jenis tanah, kelembapan, umur tanaman)."""
        v = SPEKTRAL[nama].copy()
        v[:6] *= rng.normal(1, sd_optik, 6).astype(np.float32)
        v[6:] += rng.normal(0, sd_radar, 2).astype(np.float32)
        return v
    tekstur = noise_halus(rng, 1.0)
    hutan = variasi("hutan", 0.07, 0.8)
    pre = np.empty((PATCH, PATCH, 8), np.float32)
    pre[..., :6] = hutan[:6] * (1 + 0.08 * tekstur[..., None])
    pre[..., 6:] = hutan[6:] + 0.8 * tekstur[..., None]
    if rng.random() < 0.3:
        lama = bentuk_blob(rng, rng.choice([2, 30]), rng.uniform(2, 30), rng.uniform(4, 9), 1.0)
        lama &= ~ndimage.binary_dilation(mask, iterations=2)
        pre[lama] = 0.3 * pre[lama] + 0.7 * variasi("tanaman")
    # 3) kondisi sesudah
    post = pre.copy()
    for m, nama, fraksi in lapisan:
        f = np.broadcast_to(np.asarray(fraksi, np.float32), (PATCH, PATCH))[m][:, None]
        post[m] = post[m] * (1 - f) + variasi(nama)[None, :] * f
    # 4) efek sensor
    s2_pre, s2_post = pre[..., :6].copy(), post[..., :6].copy()
    for img in (s2_pre, s2_post):
        img *= rng.normal(1, 0.05, 6).astype(np.float32)          # variasi radiometrik/atmosfer
        img += rng.normal(0, 0.008, img.shape).astype(np.float32)  # derau sensor
        np.clip(img, 0.001, 1, out=img)
    offset_medan = rng.normal(0, 1.0)                              # efek sudut datang & lereng
    s1_pre = (_speckle(pre[..., 6:], rng, 4) + offset_medan).astype(np.float32)
    s1_post = (_speckle(post[..., 6:], rng, 4) + offset_medan).astype(np.float32)
    # 4b) awan pada citra optik sesudah (radar tidak terpengaruh)
    if tutupan_awan is None:
        if rng.random() < p_awan:
            tutupan = 1.0 if rng.random() < 0.2 else rng.uniform(0.1, 0.9)
        else:
            tutupan = 0.0
    else:
        tutupan = float(tutupan_awan)
    if tutupan <= 0:
        awan = np.zeros((PATCH, PATCH), bool)
    elif tutupan >= 1:
        awan = np.ones((PATCH, PATCH), bool)
    else:
        medan = noise_halus(rng, 4.0)
        awan = medan > np.quantile(medan, 1 - tutupan)
    s2_post_obs = s2_post.copy()
    s2_post_obs[awan] = np.nan
    # 5) deret waktu bulanan (rata-rata dalam mask alert)
    i_pre, i_post = indeks_spektral(s2_pre), indeks_spektral(s2_post)
    nilai = {"NDVI": (i_pre["NDVI"][mask].mean(), i_post["NDVI"][mask].mean(), 0.05),
             "NBR":  (i_pre["NBR"][mask].mean(),  i_post["NBR"][mask].mean(),  0.05),
             "VH":   (s1_pre[..., 1][mask].mean(), s1_post[..., 1][mask].mean(), 0.8)}
    t0 = pilih_bulan_kejadian(kelas, rng)
    dt = np.arange(N_BULAN) - t0
    r = kurva_pemulihan(kelas, dt, rng)
    musim = np.sin(2 * np.pi * TANGGAL.month.values / 12)
    ts = np.zeros((N_BULAN, 3), np.float32)
    for j, (v_pre, v_post, sig) in enumerate(nilai.values()):
        v = np.where(dt < 0, v_pre, v_post + (v_pre - v_post) * r)
        ts[:, j] = v + (0.02 if j < 2 else 0.3) * musim + rng.normal(0, sig, N_BULAN)
    basah = np.isin(TANGGAL.month.values, [11, 12, 1, 2, 3, 4])
    if 0 < p_ts_hilang < 1:
        p_bulan = np.clip(p_ts_hilang + np.where(basah, 0.15, -0.15), 0, 1)
    else:
        p_bulan = np.full(N_BULAN, float(p_ts_hilang))
    hilang = rng.random(N_BULAN) < p_bulan
    ts[hilang, :2] = np.nan                            # optik hilang saat berawan; VH tetap ada
    return {"kelas": kelas, "mask": mask, "awan": awan, "tutupan_awan": float(awan.mean()),
            "s2_pre": s2_pre, "s2_post": s2_post_obs, "s1_pre": s1_pre, "s1_post": s1_post,
            "ts": ts, "t0": t0}


def bangkitkan_dataset(n_per_kelas, rng, **kwargs):
    sampel = [buat_sampel(k, rng, **kwargs) for k in range(N_KELAS) for _ in range(n_per_kelas)]
    urutan = rng.permutation(len(sampel))
    sampel = [sampel[i] for i in urutan]
    return sampel, np.array([s["kelas"] for s in sampel])

In [ ]:
# ============================================================
# 6.4c Membangkitkan dataset utama
# ============================================================
N_PER_KELAS = 350
t_mulai = time.time()
sampel_all, y_all = bangkitkan_dataset(N_PER_KELAS, rng)
print(f"Jumlah sampel  : {len(sampel_all)}  ({N_PER_KELAS} per kelas)")
print(f"Waktu generate : {time.time() - t_mulai:.1f} detik")
print(f"Rata-rata tutupan awan citra optik sesudah: {np.mean([s['tutupan_awan'] for s in sampel_all]):.1%}")
print(f"Rata-rata bulan optik hilang pada deret waktu: "
      f"{np.mean([np.isnan(s['ts'][:, 0]).mean() for s in sampel_all]):.1%}")

ringkas = pd.DataFrame({
    "Penyebab": [KELAS[s["kelas"]] for s in sampel_all],
    "Luas (ha)": [s["mask"].sum() * LUAS_PIKSEL_HA for s in sampel_all],
    "Tutupan awan": [s["tutupan_awan"] for s in sampel_all],
})
ringkas.groupby("Penyebab").agg(Jumlah=("Luas (ha)", "size"),
                                Luas_rata2_ha=("Luas (ha)", "mean"),
                                Luas_min_ha=("Luas (ha)", "min"),
                                Luas_maks_ha=("Luas (ha)", "max"),
                                Awan_rata2=("Tutupan awan", "mean")).round(2)

### 6.5 Visualisasi contoh data per penyebab

Untuk setiap penyebab ditampilkan: citra warna asli Sentinel-2 sebelum dan sesudah (area putih = awan), perubahan NDVI (ΔNDVI), citra VH Sentinel-1 sesudah, dan mask alert.

In [ ]:
# ============================================================
# 6.5 Galeri contoh: RGB sebelum/sesudah, dNDVI, VH, mask
# ============================================================
def rgb(s2, gain=6.0):
    img = np.clip(s2[..., [2, 1, 0]] * gain, 0, 1)
    img = np.where(np.isnan(img), 1.0, img)          # awan ditampilkan putih
    return img

contoh = {}
for s in sampel_all:                                  # pilih contoh dengan awan sedikit agar jelas
    if s["kelas"] not in contoh and s["tutupan_awan"] < 0.15:
        contoh[s["kelas"]] = s
    if len(contoh) == N_KELAS:
        break

judul_kolom = ["RGB sebelum", "RGB sesudah", "ΔNDVI", "VH sesudah (dB)", "Mask alert"]
fig, ax = plt.subplots(N_KELAS, 5, figsize=(10, 2.0 * N_KELAS))
for i in range(N_KELAS):
    s = contoh[i]
    dndvi = indeks_spektral(s["s2_post"])["NDVI"] - indeks_spektral(s["s2_pre"])["NDVI"]
    panel = [rgb(s["s2_pre"]), rgb(s["s2_post"]), dndvi, s["s1_post"][..., 1], s["mask"]]
    cmap = [None, None, "RdYlGn", "gray", "Reds"]
    vmin = [None, None, -0.9, -28, 0]; vmax = [None, None, 0.2, -8, 1]
    for j in range(5):
        ax[i, j].imshow(panel[j], cmap=cmap[j], vmin=vmin[j], vmax=vmax[j])
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if i == 0:
            ax[i, j].set_title(judul_kolom[j], fontsize=9)
    ax[i, 0].set_ylabel(KELAS_SINGKAT[i], fontsize=9, rotation=0, ha="right", va="center")
plt.suptitle("Contoh patch alert per penyebab (simulasi Sentinel-2 & Sentinel-1)", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 6.5b Rata-rata deret waktu per penyebab (disejajarkan pada bulan alert)
# ============================================================
rel = np.arange(-6, 7)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for k in range(N_KELAS):
    kumpulan = np.full((0, len(rel), 3), np.nan)
    for s in (s for s in sampel_all if s["kelas"] == k):
        idx = s["t0"] + rel
        kumpulan = np.vstack([kumpulan, s["ts"][idx][None]])
    rata = np.nanmean(kumpulan, axis=0)
    for j, nama in enumerate(["NDVI", "NBR", "VH (dB)"]):
        ax[j].plot(rel, rata[:, j], marker=".", color=WARNA_KELAS[k], label=KELAS_SINGKAT[k])
for j, nama in enumerate(["NDVI", "NBR", "VH (dB)"]):
    ax[j].axvline(0, color="k", ls="--", lw=0.8)
    ax[j].set(title=f"Rata-rata {nama}", xlabel="Bulan relatif terhadap alert")
ax[0].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

# Distribusi bulan kejadian per penyebab (pola musim)
bulan = pd.DataFrame({"Penyebab": [KELAS_SINGKAT[s["kelas"]] for s in sampel_all],
                      "Bulan": [TANGGAL[s["t0"]].month for s in sampel_all]})
tab = pd.crosstab(bulan["Penyebab"], bulan["Bulan"]).reindex(KELAS_SINGKAT)
plt.figure(figsize=(9, 3.2))
sns.heatmap(tab, cmap="YlOrRd", annot=True, fmt="d", cbar=False)
plt.title("Sebaran bulan kejadian alert per penyebab (kebakaran dominan di musim kemarau)")
plt.xlabel("Bulan"); plt.ylabel(""); plt.tight_layout(); plt.show()

### 6.6 Rekayasa fitur (*feature engineering*)

Fitur dikelompokkan menjadi empat kelompok agar dapat dibandingkan kontribusinya:

| Kelompok | Contoh fitur | Sumber |
|---|---|---|
| **Bentuk** | luas (ha), jumlah komponen, kekompakan $4\pi A/P^2$, elongasi, rasio isi *bounding box* | Mask alert |
| **Optik** | NDVI/NBR/NDWI/NDMI sesudah, ΔNDVI, ΔNBR, keragaman ΔNDVI, fraksi air optik, penurunan & pemulihan NDVI/NBR | Sentinel-2 |
| **Radar** | VV/VH sesudah, ΔVV, ΔVH, selisih VH–VV, fraksi air radar, penurunan & pemulihan VH | Sentinel-1 |
| **Musim** | sin/cos bulan kejadian | Tanggal alert |

In [ ]:
# ============================================================
# 6.6 Ekstraksi fitur
# ============================================================
def _rata(x, m):
    v = x[m]; v = v[~np.isnan(v)]
    return float(v.mean()) if v.size else np.nan

def _std(x, m):
    v = x[m]; v = v[~np.isnan(v)]
    return float(v.std()) if v.size > 1 else np.nan

def fitur_bentuk(mask):
    lab, n = ndimage.label(mask)
    A = mask.sum()
    P = (mask & ~ndimage.binary_erosion(mask)).sum()
    ys, xs = np.nonzero(mask)
    if A >= 3:
        ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([ys, xs]))))[::-1]
        elong, sebaran = np.sqrt((ev[0] + 1e-3) / (ev[1] + 1e-3)), np.sqrt(ev.sum())
    else:
        elong, sebaran = 1.0, 0.0
    bbox = (ys.max() - ys.min() + 1) * (xs.max() - xs.min() + 1)
    ukuran = np.asarray(ndimage.sum(mask, lab, range(1, n + 1)))
    return {"luas_ha": A * LUAS_PIKSEL_HA, "jumlah_komponen": n,
            "rasio_komponen_terbesar": ukuran.max() / A,
            "luas_rata2_komponen_ha": ukuran.mean() * LUAS_PIKSEL_HA,
            "kekompakan": 4 * np.pi * A / (P**2 + 1e-6), "rasio_tepi": P / A,
            "elongasi": elong, "rasio_isi_bbox": A / bbox, "sebaran_spasial": sebaran}

def fitur_optik(s):
    m = s["mask"]
    ip, iq = indeks_spektral(s["s2_pre"]), indeks_spektral(s["s2_post"])
    f = {}
    for k in ["NDVI", "NBR", "NDWI", "NDMI", "BRIGHT"]:
        f[f"{k}_sesudah"] = _rata(iq[k], m)
        f[f"d{k}"] = _rata(iq[k] - ip[k], m)
    f["std_dNDVI"] = _std(iq["NDVI"] - ip["NDVI"], m)
    f["B11_sesudah"] = _rata(s["s2_post"][..., 4], m)
    valid = m & ~np.isnan(iq["NDWI"])
    f["fraksi_air_optik"] = float((iq["NDWI"][valid] > 0).mean()) if valid.any() else np.nan
    cincin = ndimage.binary_dilation(m, iterations=3) & ~m
    f["NDVI_sekitar"] = _rata(iq["NDVI"], cincin)
    return f

def fitur_radar(s):
    m = s["mask"]
    f = {}
    for i, b in enumerate(BAND_S1):
        f[f"{b}_sesudah"] = _rata(s["s1_post"][..., i], m)
        f[f"d{b}"] = _rata(s["s1_post"][..., i] - s["s1_pre"][..., i], m)
    f["selisih_VH_VV"] = f["VH_sesudah"] - f["VV_sesudah"]
    f["std_VH_sesudah"] = _std(s["s1_post"][..., 1], m)
    f["fraksi_air_radar"] = float((s["s1_post"][..., 1][m] < -21).mean())
    return f

def fitur_temporal(ts, t0, nama):
    sebelum = np.nanmean(ts[:t0])
    awal = np.nanmean(ts[t0:t0 + 2])
    akhir = np.nanmean(ts[t0 + 4:t0 + 7])
    turun = sebelum - awal
    t = np.arange(len(ts))[t0:]; v = ts[t0:]; ok = ~np.isnan(v)
    tren = np.polyfit(t[ok], v[ok], 1)[0] if ok.sum() >= 3 else np.nan
    return {f"{nama}_turun": turun,
            f"{nama}_pulih": (akhir - awal) / (abs(turun) + 0.05),
            f"{nama}_akhir": akhir,
            f"{nama}_std_setelah": np.nanstd(v) if ok.sum() >= 2 else np.nan,
            f"{nama}_tren_setelah": tren}

def ekstrak_fitur(sampel):
    baris = []
    for s in sampel:
        f = {}
        f.update(fitur_bentuk(s["mask"]))
        f.update(fitur_optik(s))
        f.update(fitur_temporal(s["ts"][:, 0], s["t0"], "ts_NDVI"))
        f.update(fitur_temporal(s["ts"][:, 1], s["t0"], "ts_NBR"))
        f.update(fitur_radar(s))
        f.update(fitur_temporal(s["ts"][:, 2], s["t0"], "ts_VH"))
        bln = TANGGAL[s["t0"]].month
        f["bulan_sin"], f["bulan_cos"] = np.sin(2 * np.pi * bln / 12), np.cos(2 * np.pi * bln / 12)
        baris.append(f)
    return pd.DataFrame(baris)

t_mulai = time.time()
X_all = ekstrak_fitur(sampel_all)
print(f"Matriks fitur: {X_all.shape}  ({time.time() - t_mulai:.1f} detik)")

# Kelompok fitur
contoh_s = sampel_all[0]
FITUR_BENTUK = list(fitur_bentuk(contoh_s["mask"]).keys())
FITUR_MUSIM = ["bulan_sin", "bulan_cos"]
FITUR_OPTIK = list(fitur_optik(contoh_s).keys()) + [c for c in X_all.columns if c.startswith(("ts_NDVI", "ts_NBR"))]
FITUR_RADAR = list(fitur_radar(contoh_s).keys()) + [c for c in X_all.columns if c.startswith("ts_VH")]
FITUR_FUSI = FITUR_BENTUK + FITUR_OPTIK + FITUR_RADAR + FITUR_MUSIM
KELOMPOK = {**{c: "Bentuk" for c in FITUR_BENTUK}, **{c: "Optik (S2)" for c in FITUR_OPTIK},
            **{c: "Radar (S1)" for c in FITUR_RADAR}, **{c: "Musim" for c in FITUR_MUSIM}}
print(f"Fitur bentuk: {len(FITUR_BENTUK)} | optik: {len(FITUR_OPTIK)} | radar: {len(FITUR_RADAR)} | musim: {len(FITUR_MUSIM)}")
print(f"Persentase nilai kosong (akibat awan) pada fitur optik: {X_all[FITUR_OPTIK].isna().mean().mean():.1%}")
X_all.head()

In [ ]:
# Sebaran beberapa fitur kunci per penyebab
fitur_kunci = ["luas_ha", "elongasi", "kekompakan", "dNBR", "fraksi_air_radar", "ts_NDVI_pulih"]
df_plot = X_all[fitur_kunci].copy()
df_plot["Penyebab"] = [KELAS_SINGKAT[k] for k in y_all]
fig, ax = plt.subplots(2, 3, figsize=(14, 6.5))
for a, f in zip(ax.ravel(), fitur_kunci):
    sns.boxplot(data=df_plot, x="Penyebab", y=f, order=KELAS_SINGKAT, palette=WARNA_KELAS,
                ax=a, fliersize=1, linewidth=0.8)
    a.set_title(f); a.set_xlabel(""); a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

### 6.7 Pembagian data latih, validasi, dan uji

In [ ]:
# ============================================================
# 6.7 Split stratified 70 / 15 / 15
# ============================================================
idx = np.arange(len(y_all))
idx_latih, idx_sisa = train_test_split(idx, test_size=0.30, stratify=y_all, random_state=SEED)
idx_val, idx_uji = train_test_split(idx_sisa, test_size=0.50, stratify=y_all[idx_sisa], random_state=SEED)
idx_latih_rf = np.concatenate([idx_latih, idx_val])     # RF tidak butuh data validasi khusus

y_latih, y_val, y_uji = y_all[idx_latih], y_all[idx_val], y_all[idx_uji]
print(f"Latih: {len(idx_latih)} | Validasi: {len(idx_val)} | Uji: {len(idx_uji)}")
print("Distribusi kelas data uji:", np.bincount(y_uji))

### 6.8 Model 1 — Random Forest dan perbandingan sumber data

Random Forest dilatih dengan empat konfigurasi fitur untuk membuktikan manfaat fusi sensor yang dibahas dalam kuliah:
1. **Bentuk saja** (hanya geometri mask alert),
2. **Optik (Sentinel-2)** + bentuk,
3. **Radar (Sentinel-1)** + bentuk,
4. **Fusi Sentinel-1 + Sentinel-2** (semua fitur).

Nilai kosong akibat awan diisi median (`SimpleImputer`) disertai **indikator data hilang**, sehingga model tahu kapan data optik tidak tersedia. Khusus model fusi, data latih ditambah salinan 30% sampel yang seluruh fitur optiknya dikosongkan (**augmentasi *sensor dropout***). Tanpa langkah ini, model fusi jarang melihat kasus awan total saat pelatihan sehingga kinerjanya justru bisa jatuh di bawah model radar ketika citra optik hilang sepenuhnya.

In [ ]:
# ============================================================
# 6.8 Random Forest: perbandingan konfigurasi fitur
# ============================================================
def buat_rf():
    return Pipeline([
        ("imputasi", SimpleImputer(strategy="median", add_indicator=True)),
        ("rf", RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                                      n_jobs=-1, random_state=SEED)),
    ])

KONFIGURASI = {
    "Bentuk saja":        FITUR_BENTUK + FITUR_MUSIM,
    "Optik (S2)":         FITUR_BENTUK + FITUR_OPTIK + FITUR_MUSIM,
    "Radar (S1)":         FITUR_BENTUK + FITUR_RADAR + FITUR_MUSIM,
    "Fusi S1+S2":         FITUR_FUSI,
}
def dropout_sensor_optik(X, y, proporsi=0.3, seed=SEED):
    """Augmentasi 'sensor dropout': salinan sebagian data latih dengan seluruh fitur optik
    dikosongkan, meniru citra yang tertutup awan total, agar model fusi belajar
    mengandalkan radar ketika data optik tidak tersedia."""
    r = np.random.default_rng(seed)
    pilih = r.choice(len(X), int(proporsi * len(X)), replace=False)
    X_aug = X.iloc[pilih].copy()
    X_aug.loc[:, [c for c in FITUR_OPTIK if c in X_aug.columns]] = np.nan
    return pd.concat([X, X_aug], ignore_index=True), np.concatenate([y, y[pilih]])

model_rf, hasil_rf = {}, []
for nama, kolom in KONFIGURASI.items():
    t_mulai = time.time()
    X_l, y_l = X_all.iloc[idx_latih_rf][kolom], y_all[idx_latih_rf]
    if nama == "Fusi S1+S2":
        X_l, y_l = dropout_sensor_optik(X_l, y_l)
    m = buat_rf().fit(X_l, y_l)
    pred = m.predict(X_all.iloc[idx_uji][kolom])
    model_rf[nama] = m
    hasil_rf.append({"Konfigurasi": nama, "Jumlah fitur": len(kolom),
                     "Akurasi": accuracy_score(y_uji, pred),
                     "Macro-F1": f1_score(y_uji, pred, average="macro"),
                     "Waktu latih (s)": time.time() - t_mulai})
hasil_rf = pd.DataFrame(hasil_rf)
display(hasil_rf.round(3))

ax = hasil_rf.set_index("Konfigurasi")[["Akurasi", "Macro-F1"]].plot.bar(
    figsize=(8, 3.5), color=["#4c72b0", "#dd8452"], rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
ax.set_ylim(0, 1.05); ax.set_title("Random Forest: pengaruh sumber data terhadap kinerja (data uji)")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 6.8b Evaluasi rinci model Random Forest fusi
# ============================================================
rf_fusi = model_rf["Fusi S1+S2"]
proba_rf_uji = rf_fusi.predict_proba(X_all.iloc[idx_uji][FITUR_FUSI])
pred_rf_uji = proba_rf_uji.argmax(1)
print(classification_report(y_uji, pred_rf_uji, target_names=KELAS, digits=3))

def plot_confusion(y_true, y_pred, judul):
    cm = confusion_matrix(y_true, y_pred, labels=range(N_KELAS))
    cm_norm = cm / cm.sum(1, keepdims=True)
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1,
                xticklabels=KELAS_SINGKAT, yticklabels=KELAS_SINGKAT, cbar_kws={"label": "Proporsi recall"})
    plt.xlabel("Prediksi"); plt.ylabel("Sebenarnya"); plt.title(judul)
    plt.tight_layout(); plt.show()

plot_confusion(y_uji, pred_rf_uji, "Confusion matrix — Random Forest fusi S1+S2 (angka = jumlah sampel)")

In [ ]:
# ============================================================
# 6.8c Kepentingan fitur (permutation importance pada data uji)
# ============================================================
t_mulai = time.time()
pi = permutation_importance(rf_fusi, X_all.iloc[idx_uji][FITUR_FUSI], y_uji, n_repeats=5,
                            random_state=SEED, n_jobs=-1, scoring="f1_macro")
imp = pd.DataFrame({"Fitur": FITUR_FUSI, "Penurunan Macro-F1": pi.importances_mean,
                    "Kelompok": [KELOMPOK[c] for c in FITUR_FUSI]}).sort_values("Penurunan Macro-F1", ascending=False)
print(f"Selesai dalam {time.time() - t_mulai:.1f} detik")

warna_kel = {"Bentuk": "#8172b3", "Optik (S2)": "#55a868", "Radar (S1)": "#4c72b0", "Musim": "#c44e52"}
top = imp.head(20).iloc[::-1]
fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2, 1]})
ax[0].barh(top["Fitur"], top["Penurunan Macro-F1"], color=[warna_kel[k] for k in top["Kelompok"]])
ax[0].set_title("20 fitur terpenting (permutation importance)")
ax[0].legend(handles=[mpatches.Patch(color=v, label=k) for k, v in warna_kel.items()], fontsize=8)
per_kel = imp.groupby("Kelompok")["Penurunan Macro-F1"].sum().sort_values()
ax[1].barh(per_kel.index, per_kel.values, color=[warna_kel[k] for k in per_kel.index])
ax[1].set_title("Total kepentingan per kelompok fitur")
plt.tight_layout(); plt.show()

### 6.9 Model 2 — CNN multimodal (Deep Learning)

Model *deep learning* menerima data mentah tanpa rekayasa fitur manual, mirip pendekatan Slagter dkk. (2023):
- **Cabang citra (Conv2D)** — input 32×32×9 kanal: ΔNDVI, ΔNBR, NDWI sesudah, NDMI sesudah, ΔVV, ΔVH, VH sesudah, mask awan, mask alert.
- **Cabang deret waktu (Conv1D)** — input 24×5: NDVI, NBR, VH, indikator bulan berawan, indikator sesudah alert.
- Kedua cabang digabung (*concatenate*) lalu masuk lapisan *dense* dan *softmax* 8 kelas.

Augmentasi data: (1) rotasi 90°/180° dan pencerminan, karena orientasi bukaan di lapangan tidak menentukan penyebabnya; (2) *sensor dropout*, yaitu salinan sebagian data dengan kanal optik dikosongkan agar CNN tetap andal saat awan total.

In [ ]:
# ============================================================
# 6.9a Menyiapkan tensor input CNN
# ============================================================
KANAL_CITRA = ["dNDVI", "dNBR", "NDWI_sesudah", "NDMI_sesudah", "dVV", "dVH", "VH_sesudah", "mask_awan", "mask_alert"]
N_KANAL_NORM = 7          # 7 kanal pertama dinormalisasi, 2 kanal mask tetap 0/1

def tensor_mentah(sampel):
    n = len(sampel)
    X_img = np.zeros((n, PATCH, PATCH, len(KANAL_CITRA)), np.float32)
    X_ts = np.zeros((n, N_BULAN, 5), np.float32)
    for i, s in enumerate(sampel):
        ip, iq = indeks_spektral(s["s2_pre"]), indeks_spektral(s["s2_post"])
        X_img[i] = np.stack([iq["NDVI"] - ip["NDVI"], iq["NBR"] - ip["NBR"], iq["NDWI"], iq["NDMI"],
                             s["s1_post"][..., 0] - s["s1_pre"][..., 0],
                             s["s1_post"][..., 1] - s["s1_pre"][..., 1],
                             s["s1_post"][..., 1], s["awan"], s["mask"]], -1)
        X_ts[i, :, :3] = s["ts"]
        X_ts[i, :, 3] = np.isnan(s["ts"][:, 0])
        X_ts[i, :, 4] = np.arange(N_BULAN) >= s["t0"]
    return X_img, X_ts

def hitung_statistik(X_img, X_ts):
    return {"img_mean": np.nanmean(X_img[..., :N_KANAL_NORM], axis=(0, 1, 2)),
            "img_std": np.nanstd(X_img[..., :N_KANAL_NORM], axis=(0, 1, 2)) + 1e-6,
            "ts_mean": np.nanmean(X_ts[..., :3], axis=(0, 1)),
            "ts_std": np.nanstd(X_ts[..., :3], axis=(0, 1)) + 1e-6}

def normalisasi(X_img, X_ts, st):
    X_img, X_ts = X_img.copy(), X_ts.copy()
    X_img[..., :N_KANAL_NORM] = (X_img[..., :N_KANAL_NORM] - st["img_mean"]) / st["img_std"]
    X_ts[..., :3] = (X_ts[..., :3] - st["ts_mean"]) / st["ts_std"]
    return np.nan_to_num(X_img, nan=0.0), np.nan_to_num(X_ts, nan=0.0)   # piksel/bulan berawan = 0 (nilai rata-rata)

def siapkan_tensor(sampel, st):
    return normalisasi(*tensor_mentah(sampel), st)

X_img_raw, X_ts_raw = tensor_mentah(sampel_all)
STAT = hitung_statistik(X_img_raw[idx_latih], X_ts_raw[idx_latih])
X_img_all, X_ts_all = normalisasi(X_img_raw, X_ts_raw, STAT)
del X_img_raw, X_ts_raw

def augmentasi(X_img, X_ts, y, proporsi_dropout=0.3, seed=SEED):
    """(1) Rotasi 90°, 180°, dan pencerminan -> 4x lipat.
    (2) Sensor dropout: salinan 30% data dengan kanal optik dikosongkan (awan total)."""
    citra = [X_img, np.rot90(X_img, 1, axes=(1, 2)), np.rot90(X_img, 2, axes=(1, 2)), X_img[:, :, ::-1]]
    Xi, Xt, ya = np.concatenate(citra), np.concatenate([X_ts] * 4), np.concatenate([y] * 4)
    r = np.random.default_rng(seed)
    pilih = r.choice(len(Xi), int(proporsi_dropout * len(X_img)), replace=False)
    Xi_d, Xt_d = Xi[pilih].copy(), Xt[pilih].copy()
    Xi_d[..., 0:4] = 0.0          # kanal optik (dNDVI, dNBR, NDWI, NDMI) = nilai rata-rata
    Xi_d[..., 7] = 1.0            # mask awan = seluruh patch berawan
    Xt_d[..., 0:2] = 0.0          # deret waktu NDVI & NBR hilang
    Xt_d[..., 3] = 1.0            # indikator bulan berawan
    return (np.concatenate([Xi, Xi_d]).astype(np.float32), np.concatenate([Xt, Xt_d]).astype(np.float32),
            np.concatenate([ya, ya[pilih]]))

Xi_latih, Xt_latih, ya_latih = augmentasi(X_img_all[idx_latih], X_ts_all[idx_latih], y_latih)
print("Citra latih (augmentasi):", Xi_latih.shape, "| deret waktu:", Xt_latih.shape)
print("Citra validasi:", X_img_all[idx_val].shape, "| citra uji:", X_img_all[idx_uji].shape)

In [ ]:
# ============================================================
# 6.9b Arsitektur dan pelatihan CNN multimodal (TensorFlow/Keras)
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

def bangun_cnn(n_kanal, n_langkah, n_fitur_ts, n_kelas):
    in_citra = keras.Input(shape=(PATCH, PATCH, n_kanal), name="citra")
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(in_citra)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)                                   # 16x16
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)                                   # 8x8
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)

    in_ts = keras.Input(shape=(n_langkah, n_fitur_ts), name="deret_waktu")
    t = layers.Conv1D(32, 3, padding="same", activation="relu")(in_ts)
    t = layers.Conv1D(64, 3, padding="same", activation="relu")(t)
    t = layers.GlobalAveragePooling1D()(t)

    z = layers.Concatenate()([x, t])
    z = layers.Dense(128, activation="relu")(z)
    z = layers.Dropout(0.3)(z)
    keluaran = layers.Dense(n_kelas, activation="softmax", name="penyebab")(z)
    model = keras.Model(inputs=[in_citra, in_ts], outputs=keluaran, name="CNN_multimodal_S1_S2")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model_cnn = bangun_cnn(len(KANAL_CITRA), N_BULAN, Xt_latih.shape[-1], N_KELAS)
model_cnn.summary()

callback = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)]
t_mulai = time.time()
riwayat = model_cnn.fit([Xi_latih, Xt_latih], ya_latih,
                        validation_data=([X_img_all[idx_val], X_ts_all[idx_val]], y_val),
                        epochs=30, batch_size=64, callbacks=callback, verbose=2)
print(f"Waktu pelatihan CNN: {time.time() - t_mulai:.0f} detik")

def proba_cnn(X_img, X_ts):
    return model_cnn.predict([X_img, X_ts], batch_size=256, verbose=0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(riwayat.history["loss"], label="latih"); ax[0].plot(riwayat.history["val_loss"], label="validasi")
ax[0].set(title="Loss", xlabel="Epoch"); ax[0].legend()
ax[1].plot(riwayat.history["accuracy"], label="latih"); ax[1].plot(riwayat.history["val_accuracy"], label="validasi")
ax[1].set(title="Akurasi", xlabel="Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 6.9c Evaluasi CNN dan ensemble (rata-rata probabilitas RF + CNN)
# ============================================================
proba_cnn_uji = proba_cnn(X_img_all[idx_uji], X_ts_all[idx_uji])
pred_cnn_uji = proba_cnn_uji.argmax(1)
proba_ens_uji = (proba_rf_uji + proba_cnn_uji) / 2
pred_ens_uji = proba_ens_uji.argmax(1)

perbandingan = pd.DataFrame([
    {"Model": nama, "Akurasi": accuracy_score(y_uji, p), "Macro-F1": f1_score(y_uji, p, average="macro")}
    for nama, p in [("Random Forest (fusi)", pred_rf_uji), ("CNN multimodal", pred_cnn_uji),
                    ("Ensemble RF + CNN", pred_ens_uji)]])
display(perbandingan.round(3))

f1_per_kelas = pd.DataFrame({
    "Random Forest": f1_score(y_uji, pred_rf_uji, average=None),
    "CNN": f1_score(y_uji, pred_cnn_uji, average=None),
    "Ensemble": f1_score(y_uji, pred_ens_uji, average=None)}, index=KELAS_SINGKAT)
ax = f1_per_kelas.plot.bar(figsize=(11, 3.6), rot=0, color=["#4c72b0", "#dd8452", "#55a868"])
ax.set_ylim(0, 1.05); ax.set_title("F1-score per penyebab (data uji)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

plot_confusion(y_uji, pred_ens_uji, "Confusion matrix — Ensemble RF + CNN")

### 6.10 Uji ketahanan terhadap tutupan awan

Dibangkitkan lima set uji baru dengan tutupan awan citra optik sesudah 0%, 25%, 50%, 75%, dan 100% (peluang bulan optik hilang pada deret waktu disamakan). Model yang hanya mengandalkan Sentinel-2 diperkirakan menurun tajam, sedangkan model radar dan fusi seharusnya lebih stabil. Ini adalah alasan utama penggunaan Sentinel-1 di wilayah tropis.

In [ ]:
# ============================================================
# 6.10 Akurasi vs tutupan awan
# ============================================================
rng_awan = np.random.default_rng(SEED + 1)
LEVEL_AWAN = [0.0, 0.25, 0.5, 0.75, 1.0]
hasil_awan = []
for lv in LEVEL_AWAN:
    smp, y_lv = bangkitkan_dataset(60, rng_awan, tutupan_awan=lv, p_ts_hilang=lv)
    X_lv = ekstrak_fitur(smp)
    baris = {"Tutupan awan": f"{lv:.0%}"}
    for nama in ["Optik (S2)", "Radar (S1)", "Fusi S1+S2"]:
        baris[f"RF {nama}"] = f1_score(y_lv, model_rf[nama].predict(X_lv[KONFIGURASI[nama]]), average="macro")
    p_rf = model_rf["Fusi S1+S2"].predict_proba(X_lv[FITUR_FUSI])
    p_cnn = proba_cnn(*siapkan_tensor(smp, STAT))
    baris["CNN multimodal"] = f1_score(y_lv, p_cnn.argmax(1), average="macro")
    baris["Ensemble RF + CNN"] = f1_score(y_lv, ((p_rf + p_cnn) / 2).argmax(1), average="macro")
    hasil_awan.append(baris)
hasil_awan = pd.DataFrame(hasil_awan).set_index("Tutupan awan")
display(hasil_awan.round(3))

ax = hasil_awan.plot(marker="o", figsize=(9, 4))
ax.set(ylabel="Macro-F1", ylim=(0, 1), title="Ketahanan model terhadap tutupan awan (480 alert per level)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 6.11 Penerapan: atribusi dan prioritisasi alert baru di Provinsi Bengkulu

Model ensemble diterapkan pada **160 alert baru** yang tersebar di sembilan kabupaten di Bengkulu (lokasi dan status kawasan merupakan **simulasi ilustratif**). Untuk setiap alert dihitung:

- **Penyebab terprediksi** dan **tingkat keyakinan** (probabilitas tertinggi).
- **Skor prioritas (0–100)** = 100 × (0,35 × bobot risiko penyebab + 0,30 × bobot status kawasan + 0,20 × luas ternormalisasi + 0,15 × keyakinan).
- **Kategori prioritas**: Tinggi (≥ 70), Sedang (50–70), Rendah (< 50).
- **Status verifikasi**: alert dengan keyakinan < 0,60 ditandai *wajib verifikasi lapangan*.
- **Rekomendasi tindak lanjut** sesuai jenis penyebab.

| Penyebab | Bobot risiko | Alasan |
|---|---|---|
| Pertambangan | 1,00 | Berpotensi tambang tanpa izin, kerusakan permanen dan pencemaran |
| Penebangan kayu | 0,90 | Berpotensi pembalakan liar, sering menjadi pintu masuk perambahan |
| Perkebunan skala besar | 0,85 | Konversi permanen dalam skala luas |
| Kebakaran hutan | 0,80 | Dapat meluas cepat dan berdampak asap |
| Pertanian rakyat | 0,60 | Perambahan skala kecil, perlu pendekatan sosial |
| Infrastruktur (jalan) | 0,50 | Perlu cek izin penggunaan kawasan hutan |
| Bencana alam (longsor) | 0,30 | Penyebab alami, ranah mitigasi bencana |
| Dinamika air | 0,20 | Proses alami, cukup dipantau |

| Status kawasan | Bobot |
|---|---|
| Taman Nasional | 1,00 |
| Hutan Lindung | 0,80 |
| Hutan Produksi | 0,50 |
| Areal Penggunaan Lain (APL) | 0,20 |

In [ ]:
# ============================================================
# 6.11a Membangkitkan 160 alert baru di Bengkulu (simulasi)
# ============================================================
rng_baru = np.random.default_rng(2026)
STATUS = ["Taman Nasional", "Hutan Lindung", "Hutan Produksi", "Areal Penggunaan Lain (APL)"]
# (kabupaten, bujur, lintang pusat sebaran, bobot jumlah alert, peluang status kawasan)
ZONA = [
    ("Mukomuko",         101.45, -2.62, 0.16, [0.35, 0.15, 0.30, 0.20]),
    ("Bengkulu Utara",   102.05, -3.15, 0.16, [0.25, 0.20, 0.30, 0.25]),
    ("Lebong",           102.22, -3.05, 0.14, [0.55, 0.20, 0.10, 0.15]),
    ("Rejang Lebong",    102.55, -3.42, 0.09, [0.30, 0.35, 0.10, 0.25]),
    ("Kepahiang",        102.62, -3.62, 0.07, [0.10, 0.50, 0.10, 0.30]),
    ("Bengkulu Tengah",  102.45, -3.70, 0.08, [0.00, 0.40, 0.25, 0.35]),
    ("Seluma",           102.75, -4.00, 0.11, [0.00, 0.35, 0.35, 0.30]),
    ("Bengkulu Selatan", 103.00, -4.30, 0.08, [0.00, 0.40, 0.25, 0.35]),
    ("Kaur",             103.35, -4.62, 0.11, [0.35, 0.25, 0.15, 0.25]),
]
P_PENYEBAB = [0.18, 0.32, 0.12, 0.07, 0.07, 0.09, 0.09, 0.06]   # komposisi penyebab (simulasi)
N_ALERT = 160

bobot_zona = np.array([z[3] for z in ZONA]); bobot_zona /= bobot_zona.sum()
sampel_baru, meta = [], []
for i in range(N_ALERT):
    z = ZONA[rng_baru.choice(len(ZONA), p=bobot_zona)]
    kelas = int(rng_baru.choice(N_KELAS, p=P_PENYEBAB))
    s = buat_sampel(kelas, rng_baru)
    sampel_baru.append(s)
    tgl = TANGGAL[s["t0"]] + pd.Timedelta(days=int(rng_baru.integers(0, 28)))
    meta.append({"id_alert": f"BKL-{i + 1:04d}", "kabupaten": z[0],
                 "bujur": round(z[1] + rng_baru.normal(0, 0.05), 5),
                 "lintang": round(z[2] + rng_baru.normal(0, 0.05), 5),
                 "status_kawasan": STATUS[rng_baru.choice(4, p=z[4])],
                 "tanggal_alert": tgl.date(), "luas_ha": round(s["mask"].sum() * LUAS_PIKSEL_HA, 2),
                 "tutupan_awan": round(s["tutupan_awan"], 2),
                 "penyebab_sebenarnya": KELAS[kelas]})       # hanya untuk simulasi verifikasi lapangan
alert = pd.DataFrame(meta)
print(f"{len(alert)} alert baru | total luas {alert['luas_ha'].sum():.1f} ha")
alert.head()

In [ ]:
# ============================================================
# 6.11b Prediksi penyebab, keyakinan, skor prioritas, rekomendasi
# ============================================================
X_baru = ekstrak_fitur(sampel_baru)
p_rf_baru = rf_fusi.predict_proba(X_baru[FITUR_FUSI])
p_cnn_baru = proba_cnn(*siapkan_tensor(sampel_baru, STAT))
p_ens_baru = (p_rf_baru + p_cnn_baru) / 2

alert["penyebab_prediksi"] = [KELAS[k] for k in p_ens_baru.argmax(1)]
alert["keyakinan"] = p_ens_baru.max(1).round(3)
urut = np.sort(p_ens_baru, axis=1)
alert["kandidat_kedua"] = [KELAS[k] for k in np.argsort(p_ens_baru, axis=1)[:, -2]]
alert["margin"] = (urut[:, -1] - urut[:, -2]).round(3)

BOBOT_PENYEBAB = {"Pertambangan": 1.00, "Penebangan kayu": 0.90, "Perkebunan skala besar": 0.85,
                  "Kebakaran hutan": 0.80, "Pertanian rakyat": 0.60, "Infrastruktur (jalan)": 0.50,
                  "Bencana alam (longsor)": 0.30, "Dinamika air": 0.20}
BOBOT_STATUS = {"Taman Nasional": 1.00, "Hutan Lindung": 0.80, "Hutan Produksi": 0.50,
                "Areal Penggunaan Lain (APL)": 0.20}
TINDAK_LANJUT = {
    "Pertambangan": "Patroli gabungan Polhut & Gakkum; dugaan tambang tanpa izin",
    "Penebangan kayu": "Patroli Polhut; periksa izin pemanfaatan hutan & jalur angkut kayu",
    "Perkebunan skala besar": "Periksa perizinan usaha perkebunan & kesesuaian fungsi kawasan",
    "Kebakaran hutan": "Koordinasi Manggala Agni/BPBD; cek titik panas & pemadaman",
    "Pertanian rakyat": "Pendekatan sosial: penyuluhan, skema Perhutanan Sosial",
    "Infrastruktur (jalan)": "Periksa Persetujuan Penggunaan Kawasan Hutan (PPKH)",
    "Bencana alam (longsor)": "Koordinasi BPBD; pemetaan rawan longsor & rehabilitasi",
    "Dinamika air": "Pemantauan hidrologi; tidak memerlukan patroli",
}
AMBANG_KEYAKINAN = 0.60

luas_norm = np.clip(np.log1p(alert["luas_ha"]) / np.log1p(40), 0, 1)
alert["skor_prioritas"] = (100 * (0.35 * alert["penyebab_prediksi"].map(BOBOT_PENYEBAB)
                                 + 0.30 * alert["status_kawasan"].map(BOBOT_STATUS)
                                 + 0.20 * luas_norm + 0.15 * alert["keyakinan"])).round(1)
alert["kategori_prioritas"] = pd.cut(alert["skor_prioritas"], [-1, 50, 70, 101],
                                     labels=["Rendah", "Sedang", "Tinggi"], right=False).astype(str)
alert["status_verifikasi"] = np.where(alert["keyakinan"] < AMBANG_KEYAKINAN,
                                      "Wajib verifikasi lapangan (keyakinan rendah)",
                                      "Verifikasi sesuai jadwal prioritas")
alert["rekomendasi"] = alert["penyebab_prediksi"].map(TINDAK_LANJUT)
alert = alert.sort_values("skor_prioritas", ascending=False).reset_index(drop=True)
alert.insert(0, "peringkat", np.arange(1, len(alert) + 1))

print(alert["kategori_prioritas"].value_counts().reindex(["Tinggi", "Sedang", "Rendah"]).to_string())
print(f"Alert berkeyakinan rendah (< {AMBANG_KEYAKINAN}): {(alert['keyakinan'] < AMBANG_KEYAKINAN).sum()}")
kolom_tampil = ["peringkat", "id_alert", "kabupaten", "status_kawasan", "tanggal_alert", "luas_ha",
                "penyebab_prediksi", "keyakinan", "skor_prioritas", "kategori_prioritas", "rekomendasi"]
print("\n15 alert dengan prioritas tertinggi:")
alert[kolom_tampil].head(15)

In [ ]:
# ============================================================
# 6.11c Simulasi hasil verifikasi lapangan
# ============================================================
# Pada simulasi, penyebab sebenarnya diketahui sehingga kita dapat menilai
# seberapa berguna nilai keyakinan untuk menentukan alert yang wajib dicek.
benar = alert["penyebab_prediksi"] == alert["penyebab_sebenarnya"]
yakin = alert["keyakinan"] >= AMBANG_KEYAKINAN
ringkas_verif = pd.DataFrame({
    "Kelompok": [f"Keyakinan >= {AMBANG_KEYAKINAN}", f"Keyakinan < {AMBANG_KEYAKINAN}", "Semua alert"],
    "Jumlah alert": [yakin.sum(), (~yakin).sum(), len(alert)],
    "Prediksi benar": [benar[yakin].sum(), benar[~yakin].sum(), benar.sum()],
})
ringkas_verif["Akurasi"] = (ringkas_verif["Prediksi benar"] / ringkas_verif["Jumlah alert"]).round(3)
display(ringkas_verif)

salah = alert.loc[~benar, ["id_alert", "penyebab_sebenarnya", "penyebab_prediksi", "keyakinan", "kandidat_kedua"]]
print(f"Contoh alert yang salah diklasifikasikan ({len(salah)} alert):")
salah.head(10)

### 6.12 Peta interaktif dan ekspor data

Peta menggunakan pustaka **Folium** (Leaflet) dengan dua pilihan latar: OpenStreetMap dan citra satelit Esri. Warna isi lingkaran menunjukkan penyebab terprediksi, warna tepi menunjukkan kategori prioritas, dan ukuran lingkaran sebanding dengan luas bukaan. Klik titik untuk melihat detail alert.

In [ ]:
# ============================================================
# 6.12a Peta interaktif Folium
# ============================================================
import folium

WARNA_PRIORITAS = {"Tinggi": "#b30000", "Sedang": "#ff8c00", "Rendah": "#2b8cbe"}
peta = folium.Map(location=[-3.65, 102.35], zoom_start=8, tiles="OpenStreetMap", control_scale=True)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Citra satelit (Esri)").add_to(peta)

for kategori in ["Tinggi", "Sedang", "Rendah"]:
    grup = folium.FeatureGroup(name=f"Prioritas {kategori}")
    for _, r in alert[alert["kategori_prioritas"] == kategori].iterrows():
        warna = WARNA_KELAS[KELAS.index(r["penyebab_prediksi"])]
        info = (f"<b>{r['id_alert']}</b> (peringkat {r['peringkat']})<br>"
                f"Kabupaten: {r['kabupaten']}<br>Status kawasan: {r['status_kawasan']}<br>"
                f"Tanggal alert: {r['tanggal_alert']}<br>Luas: {r['luas_ha']} ha<br>"
                f"<b>Penyebab: {r['penyebab_prediksi']}</b> (keyakinan {r['keyakinan']:.2f})<br>"
                f"Skor prioritas: {r['skor_prioritas']} ({kategori})<br>"
                f"{r['status_verifikasi']}<br><i>{r['rekomendasi']}</i>")
        folium.CircleMarker(location=[r["lintang"], r["bujur"]], radius=4 + np.sqrt(r["luas_ha"]) * 1.5,
                            color=WARNA_PRIORITAS[kategori], weight=2.5, fill=True, fill_color=warna,
                            fill_opacity=0.85, popup=folium.Popup(info, max_width=320),
                            tooltip=f"{r['id_alert']} - {r['penyebab_prediksi']}").add_to(grup)
    grup.add_to(peta)
folium.LayerControl(collapsed=False).add_to(peta)

legenda = "".join(f'<div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;'
                  f'background:{w};margin-right:6px"></span>{k}</div>' for k, w in zip(KELAS, WARNA_KELAS))
legenda_html = (f'<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;'
                f'padding:8px 10px;border:1px solid #999;border-radius:6px;font-size:12px">'
                f'<b>Penyebab terprediksi</b>{legenda}<div style="margin-top:4px;font-size:11px">'
                f'Tepi: merah = prioritas tinggi, oranye = sedang, biru = rendah</div></div>')
peta.get_root().html.add_child(folium.Element(legenda_html))
peta.save("peta_prioritas_alert_bengkulu.html")
peta

In [ ]:
# ============================================================
# 6.12b Ekspor hasil ke CSV dan GeoJSON
# ============================================================
kolom_ekspor = [c for c in alert.columns if c != "penyebab_sebenarnya"]   # label simulasi tidak diekspor
alert[kolom_ekspor].to_csv("hasil_atribusi_alert_bengkulu.csv", index=False)

fitur_geo = []
for _, r in alert[kolom_ekspor].iterrows():
    prop = {k: (str(v) if k == "tanggal_alert" else (v.item() if hasattr(v, "item") else v))
            for k, v in r.items() if k not in ("bujur", "lintang")}
    fitur_geo.append({"type": "Feature", "properties": prop,
                      "geometry": {"type": "Point", "coordinates": [float(r["bujur"]), float(r["lintang"])]}})
with open("hasil_atribusi_alert_bengkulu.geojson", "w", encoding="utf-8") as f:
    json.dump({"type": "FeatureCollection", "features": fitur_geo}, f, ensure_ascii=False, indent=1)

print("File tersimpan (lihat panel Files di sebelah kiri Colab):")
for nama in ["hasil_atribusi_alert_bengkulu.csv", "hasil_atribusi_alert_bengkulu.geojson",
             "peta_prioritas_alert_bengkulu.html"]:
    print("  -", nama)

### 6.13 Dasbor ringkasan pemantauan

Dasbor statis ini merangkum informasi yang dibutuhkan pengambil keputusan: komposisi penyebab, luas terdampak, sebaran prioritas per kabupaten, sebaran spasial, dan distribusi tingkat keyakinan model.

In [ ]:
# ============================================================
# 6.13 Dasbor ringkasan
# ============================================================
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# (1) Jumlah alert per penyebab
ax1 = fig.add_subplot(gs[0, 0])
jml = alert["penyebab_prediksi"].value_counts().reindex(KELAS).fillna(0)
ax1.barh(KELAS_SINGKAT, jml.values, color=WARNA_KELAS); ax1.invert_yaxis()
ax1.bar_label(ax1.containers[0], fontsize=8); ax1.set_title("Jumlah alert per penyebab")

# (2) Luas per penyebab
ax2 = fig.add_subplot(gs[0, 1])
luas = alert.groupby("penyebab_prediksi")["luas_ha"].sum().reindex(KELAS).fillna(0)
ax2.barh(KELAS_SINGKAT, luas.values, color=WARNA_KELAS); ax2.invert_yaxis()
ax2.bar_label(ax2.containers[0], fmt="%.0f", fontsize=8); ax2.set_title("Total luas bukaan (ha)")

# (3) Distribusi keyakinan
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(alert["keyakinan"], bins=20, color="#4c72b0", edgecolor="white")
ax3.axvline(AMBANG_KEYAKINAN, color="red", ls="--", label=f"ambang {AMBANG_KEYAKINAN}")
ax3.set(title="Distribusi keyakinan model", xlabel="Keyakinan"); ax3.legend()

# (4) Prioritas per kabupaten
ax4 = fig.add_subplot(gs[1, 0:2])
pk = pd.crosstab(alert["kabupaten"], alert["kategori_prioritas"]).reindex(columns=["Tinggi", "Sedang", "Rendah"], fill_value=0)
pk = pk.loc[pk.sum(1).sort_values(ascending=False).index]
pk.plot.bar(stacked=True, ax=ax4, color=["#b30000", "#ff8c00", "#2b8cbe"], rot=30)
ax4.set(title="Kategori prioritas per kabupaten", xlabel="", ylabel="Jumlah alert")

# (5) Sebaran spasial
ax5 = fig.add_subplot(gs[1, 2])
for k, w in zip(KELAS, WARNA_KELAS):
    d = alert[alert["penyebab_prediksi"] == k]
    ax5.scatter(d["bujur"], d["lintang"], s=10 + d["skor_prioritas"] * 0.8, c=w, alpha=0.8,
                edgecolor="k", linewidth=0.3)
for z in ZONA:
    ax5.annotate(z[0], (z[1], z[2]), fontsize=7, xytext=(6, 0), textcoords="offset points")
ax5.set(title="Sebaran spasial (ukuran = skor prioritas)", xlabel="Bujur", ylabel="Lintang")

fig.suptitle(f"Dasbor Atribusi Penyebab Alert Deforestasi — Provinsi Bengkulu (simulasi, {len(alert)} alert)",
             fontsize=13, fontweight="bold")
plt.show()

print(f"Total alert                  : {len(alert)}")
print(f"Total luas bukaan            : {alert['luas_ha'].sum():.1f} ha")
print(f"Alert di Taman Nasional      : {(alert['status_kawasan'] == 'Taman Nasional').sum()}")
print(f"Prioritas tinggi             : {(alert['kategori_prioritas'] == 'Tinggi').sum()}")
print(f"Wajib verifikasi (keyakinan rendah): {(alert['keyakinan'] < AMBANG_KEYAKINAN).sum()}")
print(f"Penyebab alami (longsor + air): "
      f"{alert['penyebab_prediksi'].isin(['Bencana alam (longsor)', 'Dinamika air']).sum()} alert "
      f"-> dapat dialihkan ke BPBD / pemantauan tanpa patroli")

In [ ]:
# ============================================================
# 6.14 Ringkasan angka hasil eksperimen (otomatis)
# ============================================================
r = hasil_rf.set_index("Konfigurasi")
p = perbandingan.set_index("Model")
print("RINGKASAN HASIL")
print("=" * 60)
for nama in KONFIGURASI:
    print(f"RF {nama:<14}: akurasi {r.loc[nama, 'Akurasi']:.3f} | Macro-F1 {r.loc[nama, 'Macro-F1']:.3f}")
for nama in p.index:
    print(f"{nama:<17}: akurasi {p.loc[nama, 'Akurasi']:.3f} | Macro-F1 {p.loc[nama, 'Macro-F1']:.3f}")
print("-" * 60)
print(f"Macro-F1 pada awan 100%: RF optik {hasil_awan.iloc[-1]['RF Optik (S2)']:.3f} | "
      f"RF fusi {hasil_awan.iloc[-1]['RF Fusi S1+S2']:.3f} | ensemble {hasil_awan.iloc[-1]['Ensemble RF + CNN']:.3f}")
terbaik = f1_per_kelas["Ensemble"].idxmax(); terburuk = f1_per_kelas["Ensemble"].idxmin()
print(f"Kelas termudah (F1 ensemble): {terbaik} ({f1_per_kelas['Ensemble'].max():.3f})")
print(f"Kelas tersulit (F1 ensemble): {terburuk} ({f1_per_kelas['Ensemble'].min():.3f})")
print(f"Akurasi alert berkeyakinan tinggi vs rendah: "
      f"{ringkas_verif.loc[0, 'Akurasi']:.3f} vs {ringkas_verif.loc[1, 'Akurasi']:.3f}")

## 7. Analisis Hasil

**1) Fusi Sentinel-1 dan Sentinel-2 memberikan kinerja terbaik.**
Model yang hanya memakai bentuk mask alert (Macro-F1 ± 0,78) sudah dapat mengenali penyebab dengan geometri khas (jalan yang linier, longsor yang memanjang, perkebunan berbentuk blok), tetapi masih keliru pada penyebab yang bentuknya mirip. Penambahan informasi radar (± 0,88) dan terutama spektral optik (± 0,94) meningkatkan kinerja secara nyata. Pada data uji yang sebagian besar berawan ringan, model fusi setara atau sedikit di atas model optik (± 0,94); keunggulan utamanya muncul pada kondisi berawan (poin 2). Hal ini sejalan dengan materi kuliah bahwa setiap sensor melihat aspek permukaan yang berbeda: optik peka terhadap kondisi vegetasi dan bekas bakar, radar peka terhadap struktur permukaan dan air, serta tidak terhalang awan.

**2) Radar menjaga kinerja saat awan tebal.**
Pada uji ketahanan, model optik saja masih baik pada awan rendah–sedang tetapi turun drastis ketika awan menutupi 100% citra, karena seluruh fitur spektral dan deret waktu optik hilang. Model radar tidak terpengaruh awan. Model fusi (yang dilatih dengan augmentasi *sensor dropout*) unggul pada awan rendah–sedang dan tetap mendekati kinerja model radar ketika awan total. Untuk Bengkulu yang curah hujannya tinggi, Sentinel-1 bukan sekadar pelengkap, melainkan komponen wajib.

**3) Fitur yang paling menentukan.**
*Permutation importance* model fusi didominasi fitur **bentuk** (rasio tepi, luas, ukuran komponen, kekompakan, elongasi) dan fitur **optik** (keragaman ΔNDVI, perubahan kecerahan, NDWI). Fitur radar tampak kecil kepentingannya pada model fusi bukan karena tidak berguna, melainkan karena informasinya tumpang tindih dengan fitur optik ketika citra optik tersedia; *permutation importance* memang cenderung meremehkan fitur yang saling berkorelasi. Peran radar baru terlihat jelas pada uji awan (Bagian 6.10), saat model radar saja tetap stabil.

**4) Kelas yang sulit dibedakan.**
Pada simulasi ini, **pertanian rakyat** menjadi kelas tersulit karena variasinya paling besar: kadang dibuka dengan membakar (mirip kebakaran), kadang berbentuk petak persegi (mirip blok perkebunan), dan kadang disertai jalan desa (mirip infrastruktur). **Pertambangan** tanpa kolam air juga sering tertukar dengan perkebunan atau pertanian karena sama-sama berupa tanah terbuka. Sebaliknya, pada data riil Slagter dkk. (2023), pertanian rakyat justru paling akurat, sedangkan jalan, penebangan selektif, dan tambang lebih sulit dibedakan. Perbedaan ini menunjukkan bahwa tingkat kesulitan setiap kelas sangat dipengaruhi kondisi lapangan dan kualitas data latih, sehingga model perlu dievaluasi ulang dengan data riil Bengkulu.

**5) Random Forest vs CNN.**
Random Forest bekerja baik karena fiturnya dirancang dengan pengetahuan domain, cepat dilatih, dan mudah dijelaskan. CNN belajar langsung dari citra dan deret waktu tanpa rekayasa fitur manual; dengan data latih lebih banyak dan citra beresolusi lebih tinggi, CNN berpotensi menangkap pola spasial yang lebih halus. **Ensemble** keduanya menggabungkan kelebihan masing-masing dan menghasilkan prediksi yang lebih stabil.

**6) Nilai keyakinan berguna untuk verifikasi lapangan.**
Pada simulasi verifikasi, akurasi alert berkeyakinan tinggi jauh lebih baik dibanding alert berkeyakinan rendah. Artinya, ambang keyakinan dapat dipakai untuk menentukan alert mana yang tidak boleh langsung dipercaya dan wajib dicek di lapangan, sesuai pesan kuliah bahwa hasil AI tetap perlu diverifikasi.

**7) Dampak operasional.**
Skor prioritas memungkinkan tim patroli yang terbatas fokus pada alert berisiko tinggi di Taman Nasional dan Hutan Lindung (misalnya dugaan tambang atau penebangan), sedangkan alert akibat proses alami (longsor, dinamika air) dapat dialihkan ke BPBD atau cukup dipantau. Hasil ekspor CSV/GeoJSON dapat langsung dibuka di QGIS, ArcGIS, atau aplikasi pemantauan lain.

> Nilai numerik persis dapat sedikit berbeda antarversi pustaka; angka hasil eksekusi dapat dilihat pada keluaran sel 6.8, 6.9c, 6.10, 6.11c, dan ringkasan otomatis 6.14.

## 8. Keterbatasan dan Pengembangan Lanjutan

**Keterbatasan:**
1. Data bersifat sintetis. Meskipun meniru karakteristik fisik Sentinel-1/2, kompleksitas lapangan (topografi curam Bukit Barisan, efek bayangan radar, kebun campur, dan penyebab ganda) belum sepenuhnya terwakili. Akurasi pada data riil diperkirakan lebih rendah.
2. Satu alert diasumsikan memiliki satu penyebab dominan, padahal di lapangan penyebab bisa berurutan (misalnya penebangan → pembakaran → kebun).
3. Lokasi, status kawasan, dan bobot prioritas adalah ilustrasi; penerapan nyata memerlukan peta kawasan hutan resmi dan bobot yang disepakati bersama instansi terkait.
4. Resolusi 20 m belum cukup untuk mendeteksi bukaan yang sangat kecil.

**Pengembangan lanjutan:**
1. Mengganti data sintetis dengan alert RADD/GLAD riil serta patch Sentinel-1/2 dari Google Earth Engine (lihat Lampiran), dengan label hasil interpretasi visual citra resolusi tinggi dan survei lapangan.
2. Menambahkan data pendukung: DEM/kemiringan lereng (membantu mengenali longsor), jarak ke jalan/sungai/permukiman, peta konsesi, dan titik panas (*hotspot*) untuk kebakaran.
3. Memakai arsitektur yang lebih canggih (U-Net/ResNet, Transformer temporal) dan *transfer learning*.
4. Menerapkan *active learning*: hasil verifikasi lapangan dimasukkan kembali sebagai data latih baru.
5. Membangun dasbor web (misalnya Streamlit/Flask + Leaflet) yang terhubung ke basis data PostGIS dan diperbarui otomatis setiap alert baru terbit.

## 9. Kesimpulan

1. Studi kasus ini berhasil mengimplementasikan alur kerja atribusi penyebab langsung deforestasi: dari alert, ekstraksi data citra Sentinel-1/2 dan deret waktu, rekayasa fitur, model AI (Random Forest, CNN multimodal, dan ensemble), hingga prioritisasi, peta interaktif, dasbor, dan ekspor data.
2. Setiap penyebab deforestasi memiliki tanda khas pada bentuk bukaan, respons spektral, respons radar, dan pola waktu, sehingga dapat dikenali oleh model AI dengan kinerja tinggi pada data simulasi.
3. Fusi sensor optik (Sentinel-2) dan radar (Sentinel-1) memberikan hasil terbaik, dan radar terbukti menjaga kinerja ketika citra optik tertutup awan, yaitu kondisi yang sangat umum di Bengkulu.
4. Tingkat keyakinan model efektif untuk menandai alert yang wajib diverifikasi di lapangan, sedangkan skor prioritas membantu instansi kehutanan menggunakan sumber daya patroli yang terbatas secara lebih tepat sasaran.
5. Studi kasus ini menunjukkan bahwa ilmu informatika (pengolahan citra, *machine learning*, *deep learning*, analisis geospasial, dan visualisasi data) dapat berkontribusi langsung pada perlindungan hutan dan kawasan konservasi di Provinsi Bengkulu.

## 10. Referensi

1. Slagter, B., Reiche, J., Marcos, D., Mullissa, A., Lossou, E., Peña-Claros, M., & Herold, M. (2023). Monitoring direct drivers of small-scale tropical forest disturbance in near real-time with Sentinel-1 and -2 data. *Remote Sensing of Environment*, 295, 113655. https://doi.org/10.1016/j.rse.2023.113655
2. Reiche, J., Mullissa, A., Slagter, B., Gou, Y., Tsendbazar, N.-E., Odongo-Braun, C., Vollrath, A., Weisse, M. J., Stolle, F., Pickens, A., Donchyts, G., Clinton, N., Gorelick, N., & Herold, M. (2021). Forest disturbance alerts for the Congo Basin using Sentinel-1. *Environmental Research Letters*, 16(2), 024005. https://doi.org/10.1088/1748-9326/abd0a8
3. Curtis, P. G., Slay, C. M., Harris, N. L., Tyukavina, A., & Hansen, M. C. (2018). Classifying drivers of global forest loss. *Science*, 361(6407), 1108–1111.
4. Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32.
5. LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. *Nature*, 521, 436–444.
6. European Space Agency. Copernicus Sentinel-1 dan Sentinel-2 Mission. https://sentinels.copernicus.eu
7. World Resources Institute. Global Forest Watch. https://www.globalforestwatch.org
8. Materi Kuliah Praktisi Informatika: *Attributing Direct Drivers to Deforestation Alerts with Satellite Imagery and AI*, Dr. Bart Slagter (WUR), Program Studi S1 Teknik Informatika, Universitas Bengkulu.

## 11. Lampiran: Pengambilan Data Riil dengan Google Earth Engine

Kode berikut menunjukkan cara mengganti data sintetis dengan data riil untuk wilayah Bengkulu: alert RADD (katalog komunitas GEE `projects/radar-wur/raddalert/v1`), komposit Sentinel-2 bebas awan, dan komposit Sentinel-1. Kode **tidak dijalankan secara bawaan** (`JALANKAN_GEE = False`) karena memerlukan akun Google Earth Engine dan ID proyek Google Cloud. Untuk menjalankannya, ubah menjadi `True` dan isi `ID_PROYEK_GEE`.

In [ ]:
# ============================================================
# Lampiran: contoh pengambilan data riil (Google Earth Engine)
# ============================================================
JALANKAN_GEE = False
ID_PROYEK_GEE = "isi-id-proyek-gee"

if JALANKAN_GEE:
    import ee
    ee.Authenticate()
    ee.Initialize(project=ID_PROYEK_GEE)

    # Batas Provinsi Bengkulu (FAO GAUL level 1)
    aoi = (ee.FeatureCollection("FAO/GAUL/2015/level1")
             .filter(ee.Filter.eq("ADM1_NAME", "Bengkulu")).geometry())

    # 1) Alert RADD terbaru untuk wilayah Asia
    radd = ee.ImageCollection("projects/radar-wur/raddalert/v1")
    alert_radd = ee.Image(radd.filterMetadata("layer", "contains", "alert")
                              .filterMetadata("geography", "equals", "asia")
                              .sort("system:time_end", False).first()).clip(aoi)
    # Band 'Alert': 2 = keyakinan rendah, 3 = keyakinan tinggi
    alert_tinggi = alert_radd.select("Alert").eq(3).selfMask()

    # 2) Komposit Sentinel-2 sesudah gangguan (mask awan dari band SCL)
    def mask_awan_s2(img):
        scl = img.select("SCL")
        bersih = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
        return img.select(BAND_S2).updateMask(bersih).divide(10000)

    s2_sesudah = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterBounds(aoi).filterDate("2026-01-01", "2026-07-01")
                    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
                    .map(mask_awan_s2).median())

    # 3) Komposit Sentinel-1 (VV, VH) sesudah gangguan
    s1_sesudah = (ee.ImageCollection("COPERNICUS/S1_GRD")
                    .filterBounds(aoi).filterDate("2026-01-01", "2026-07-01")
                    .filter(ee.Filter.eq("instrumentMode", "IW"))
                    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
                    .select(BAND_S1).median())

    # 4) Sampel titik alert keyakinan tinggi + nilai citra pada titik tersebut
    titik = alert_tinggi.sample(region=aoi, scale=10, numPixels=500, seed=SEED, geometries=True)
    tumpukan = s2_sesudah.addBands(s1_sesudah)
    nilai_titik = tumpukan.sampleRegions(collection=titik, scale=20, geometries=True)
    print("Jumlah titik alert sampel:", nilai_titik.size().getInfo())

    # 5) Ekspor tumpukan citra ke Google Drive untuk dipotong menjadi patch 32x32
    tugas = ee.batch.Export.image.toDrive(image=tumpukan.toFloat(), description="stack_s1_s2_bengkulu",
                                          region=aoi, scale=20, maxPixels=1e13)
    tugas.start()
    print("Ekspor dimulai; patch hasil ekspor diproses dengan fungsi ekstrak_fitur() dan siapkan_tensor().")
else:
    print("Lampiran GEE tidak dijalankan (JALANKAN_GEE = False). Notebook memakai data simulasi.")